# Notebook 03 — Avaliação da geração RAG

**Objetivo:** estabelecer uma linha de base do RAG com Gemini e `top_k=20`, reutilizando as 19 perguntas do Notebook 02. Avaliar separadamente **execução**, **abstenção**, **fidelidade factual**, **qualidade das citações** e **síntese**.

**Importante:** uma citação com ID válido não comprova que ela sustenta a afirmação. A qualidade semântica será avaliada manualmente. O notebook não modifica `src/rag.py` nem o índice.

## 1. Ambiente e parâmetros
Execute o notebook a partir do projeto. A chave fica no `.env`, nunca nas saídas. A execução em lote está **desativada por padrão** para evitar chamadas acidentais à API.

In [99]:
"""Load project dependencies and establish reproducible evaluation settings."""
from pathlib import Path
import json
import os
import time
from datetime import datetime, timezone

import chromadb
import pandas as pd
from dotenv import load_dotenv
from google import genai
from sentence_transformers import SentenceTransformer

import sys
import hashlib

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src" / "rag.py").exists():
    raise FileNotFoundError(
        f"Não foi possível localizar a raiz do projeto: {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag import (
    generate_answer, SYSTEM_INSTRUCTION, RESPONSE_SCHEMA, USER_PROMPT_TEMPLATE
)
from src.retrieve import CHROMA_PATH, COLLECTION_NAME, MODEL_NAME

load_dotenv(PROJECT_ROOT / ".env")
MODEL_ID = os.getenv("GEMINI_MODEL")
assert os.getenv("GEMINI_API_KEY"), "Configure GEMINI_API_KEY no .env"
assert MODEL_ID, "Configure GEMINI_MODEL no .env"
TOP_K = 20
PROMPT_COMPONENTS = {
    "system_instruction": SYSTEM_INSTRUCTION,
    "response_schema": RESPONSE_SCHEMA,
    "user_prompt_template": USER_PROMPT_TEMPLATE,
}
PROMPT_SHA256 = hashlib.sha256(
    json.dumps(PROMPT_COMPONENTS, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()[:12]
# Set True ONLY if you confirm all legacy runs used the current prompt.
LEGACY_PROMPT_CONFIRMED = True
EVAL_DIR = PROJECT_ROOT / "evaluation"
EVAL_DIR.mkdir(exist_ok=True)
RUNS_PATH = EVAL_DIR / "generation_runs.jsonl"
LABELS_PATH = EVAL_DIR / "generation_labels.csv"
print(f"Modelo: {MODEL_ID} | top_k={TOP_K} | prompt={PROMPT_SHA256} | pasta: {EVAL_DIR}")

Modelo: gemini-3.8-flash | top_k=20 | prompt=a128a1fe07c1 | pasta: /home/erick/Projetos/tech_challenge_4_voice_of_customer_intelligence_rag/evaluation


## 2. Perguntas de avaliação
Mesmo conjunto e mesma classificação do Notebook 02: nove de domínio, sete fora do escopo e três vagas.

In [100]:
"""Define in-domain, ambiguous and out-of-scope evaluation questions."""

questions = pd.DataFrame([
    ("Q01", "Entrega", "Quais avaliações relatam que o produto não foi entregue?", "domain"),
    ("Q02", "Entrega", "Quais clientes reclamam de atraso na entrega?", "domain"),
    ("Q03", "Entrega", "Existem relatos de produtos entregues antes do prazo?", "domain"),
    ("Q04", "Qualidade", "Quais avaliações mencionam produtos com defeito?", "domain"),
    ("Q05", "Qualidade", "Quais clientes relatam produtos diferentes do anunciado?", "domain"),
    ("Q06", "Atendimento", "Quais avaliações mencionam dificuldades com o atendimento?", "domain"),
    ("Q07", "Pós-venda", "Existem reclamações sobre devolução ou reembolso?", "domain"),
    ("Q08", "Satisfação", "Quais avaliações elogiam a qualidade dos produtos?", "domain"),
    ("Q09", "Satisfação", "Quais clientes recomendam a experiência de compra?", "domain"),
    ("Q10", "Fora do escopo", "Qual é o melhor smartphone de 2025?", "out_of_scope"),
    ("Q11", "Fora do escopo", "Qual será o faturamento da Olist em 2027?", "out_of_scope"),
    ("Q12", "Fora do escopo", "Qual é a cotação atual do dólar?", "out_of_scope"),
    ("Q13", "Fora do escopo", "Qual é a política de frete da Olist em 2025?", "out_of_scope"),
    ("Q14", "Fora do escopo", "A Olist oferece garantia estendida em 2026?", "out_of_scope"),
    ("Q15", "Fora do escopo", "Quanto custa anunciar na Olist hoje?", "out_of_scope"),
    ("Q16", "Fora do escopo", "Qual o prazo oficial de reembolso da Olist atualmente?", "out_of_scope"),
    ("Q17", "Vaga", "Entrega?", "ambiguous"),
    ("Q18", "Vaga", "Problemas com a compra", "ambiguous"),
    ("Q19", "Vaga", "Foi bom?", "ambiguous"),
], columns=["question_id", "category", "question", "scope"])

assert questions["question_id"].is_unique
display(questions)


,question_id,category,question,scope
0,Q01,Entrega,Quais avaliações relatam que o produto não foi...,domain
1,Q02,Entrega,Quais clientes reclamam de atraso na entrega?,domain
2,Q03,Entrega,Existem relatos de produtos entregues antes do...,domain
3,Q04,Qualidade,Quais avaliações mencionam produtos com defeito?,domain
4,Q05,Qualidade,Quais clientes relatam produtos diferentes do ...,domain
5,Q06,Atendimento,Quais avaliações mencionam dificuldades com o ...,domain
6,Q07,Pós-venda,Existem reclamações sobre devolução ou reembolso?,domain
7,Q08,Satisfação,Quais avaliações elogiam a qualidade dos produ...,domain
8,Q09,Satisfação,Quais clientes recomendam a experiência de com...,domain
9,Q10,Fora do escopo,Qual é o melhor smartphone de 2025?,out_of_scope


## 3. Inicializar os componentes uma única vez
O modelo de embeddings e a coleção são reutilizados em todas as consultas.

In [101]:
"""Initialize the existing persistent index and Gemini client."""
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = chroma_client.get_collection(COLLECTION_NAME)
embedding_model = SentenceTransformer(MODEL_NAME)
llm_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("Documentos no índice:", collection.count())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Documentos no índice: 42138


## 4. Executar e registrar uma consulta
Comece com Q01. Cada execução grava um registro JSONL com pergunta, resposta, status, evidências citadas, textos recuperados, modelo e tempo. **Não são armazenadas chaves de API.** Se houver falha técnica, ela permanece `status=erro`, não é contabilizada como abstenção.

In [102]:
"""Execute one question and persist the complete evidence snapshot."""
def evaluate_one(question_id: str) -> dict:
    """Run RAG once and append an auditable snapshot to JSONL."""
    selected = questions.loc[questions["question_id"] == question_id]
    if len(selected) != 1:
        raise ValueError(f"Pergunta não encontrada: {question_id}")
    item = selected.iloc[0]
    started = time.perf_counter()
    result = generate_answer(
        question=item["question"],
        collection=collection,
        embedding_model=embedding_model,
        llm_client=llm_client,
        llm_model=MODEL_ID,
        top_k=TOP_K,
    )
    record = {
        "question_id": question_id,
        "category": item["category"],
        "scope": item["scope"],
        "question": item["question"],
        "model": MODEL_ID,
        "top_k": TOP_K,
        "prompt_sha256": PROMPT_SHA256,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "status": result["status"],
        "answer": result["answer"],
        "cited_review_ids": result["cited_review_ids"],
        "insufficient_evidence": result["insufficient_evidence"],
        "insufficiency_reason": result["insufficiency_reason"],
        "limited_evidence": result["limited_evidence"],
        "error": result["error"],
        "retrieval": result["retrieval"],
    }
    with RUNS_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    return record

# Reuse Q01 only if its configuration and prompt identity are verified.
def matches_experiment(record: dict) -> bool:
    """Match model, retrieval depth and verified prompt identity."""
    if record.get("model") != MODEL_ID or record.get("top_k") != TOP_K:
        return False
    recorded_hash = record.get("prompt_sha256")
    return (recorded_hash == PROMPT_SHA256 or
            (recorded_hash is None and LEGACY_PROMPT_CONFIRMED))

existing_q01 = []
if RUNS_PATH.exists():
    with RUNS_PATH.open(encoding="utf-8") as handle:
        existing_q01 = [
            record for line in handle if line.strip()
            if (record := json.loads(line)).get("question_id") == "Q01"
            and matches_experiment(record)
            and record.get("status") != "erro"
        ]

# Do not call the API automatically if an unverified legacy Q01 exists.
if existing_q01:
    first = existing_q01[-1]
    print("Q01 reutilizada:", first["status"])
elif RUNS_PATH.exists() and any(
    r.get("question_id") == "Q01" and r.get("model") == MODEL_ID
    and r.get("top_k") == TOP_K and r.get("prompt_sha256") is None
    for r in [json.loads(line) for line in RUNS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
):
    first = None
    print("Q01 legada encontrada, mas o prompt não foi verificado. Nenhuma chamada automática.")
else:
    first = None
    print("Q01 não executada automaticamente. Ative o lote conscientemente na seção 6.")

if first is not None:
    print("Status:", first["status"], "| Tempo:", first["elapsed_seconds"], "s")
    print("Resposta:", first["answer"])
    print("Citações:", len(first["cited_review_ids"]))

Q01 reutilizada: respondida
Status: respondida | Tempo: 240.402 s
Resposta: Diversos clientes relatam expressamente que seus pedidos não foram entregues ou não haviam chegado até o momento da avaliação. Além de apontarem a ausência de recebimento da mercadoria, alguns consumidores cobram explicações sobre o ocorrido e buscam uma solução para o problema. Há também relatos de frustração por serem solicitados a avaliar uma compra antes mesmo de tê-la em mãos.
Citações: 16


## 5. Inspeção das fontes citadas
Conferir se cada texto citado **realmente sustenta** o que a resposta afirma. A verificação automática abaixo apenas confere a presença do identificador no contexto.

In [103]:
"""Display cited review texts and check citation identifier membership."""
def cited_evidence(record: dict) -> pd.DataFrame:
    """Display cited source texts from the stored retrieval snapshot."""
    reviews = record["retrieval"]["reviews"]
    by_id = {review["review_id"]: review for review in reviews}
    return pd.DataFrame([
        {
            "review_id": review_id,
            "id_found": review_id in by_id,
            "document": by_id.get(review_id, {}).get("document", ""),
        }
        for review_id in record["cited_review_ids"]
    ], columns=["review_id", "id_found", "document"])

if first is not None:
    display(cited_evidence(first))
else:
    print("Q01 não carregada: verifique o prompt legado ou execute o lote conscientemente.")

,review_id,id_found,document
0,c84ca514000aa704df0258d693a07414,True,"Por favor, peço que chequem o porque do produt..."
1,09db4c326e7158211d66b9f994542274,True,PRODUTO NÃO ENTREGUE Gostaria de saber pq o pr...
2,4d05593294cb87f9412ec522d3c6333e,True,"ainda não recebi o produto, mas sem receber o ..."
3,2f59a88f994c5d6f802f0ddab0dd785b,True,Produto não entregue O produto ainda não foi e...
4,2f0d288c4d136bc3db96726b533b70bb,True,O produto não foi entregue
5,dd1deb7faf933a2bd44dd10cdfa9a953,True,nao satisfeito o produto ainda não foi entregue
6,f9fe37d91c506b29a0b4d9107f667bde,True,"O produto foi comprado em 10/10/2016, não cheg..."
7,09b81fdc5bc3fdcebf3152e93428aac2,True,Produto não entregue O produto ainda não foi e...
8,c1df364081b02bb8c35a7480e990126d,True,Produto não entregue Gostaria de informações s...
9,e54ce4d23537baeeca74cdd163510b2a,True,O produto não foi recebido


## 6. Lote completo — somente quando você decidir
Mantenha `RUN_BATCH=False` até revisar a primeira resposta. Cada chamada pode consumir cota. A execução de Q01 também reutiliza o resultado já salvo ao reexecutar a seção 4. O código reutiliza apenas execuções bem-sucedidas com a mesma pergunta, modelo, `top_k` e versão do prompt (`prompt_sha256`). Alterações no prompt ou no schema criam uma nova versão do experimento, sem sobrescrever registros anteriores.

**Atenção:** registros antigos não têm `prompt_sha256`. Para reutilizá-los sem novas chamadas, defina `LEGACY_PROMPT_CONFIRMED=True` na seção 1 **somente após confirmar** que as instruções, o schema e o template do prompt de `src/rag.py` não mudaram desde as execuções. Erros técnicos são reexecutáveis quando o lote estiver ativado.


In [104]:
"""Optionally run only questions missing from the current experiment."""
RUN_BATCH = False

def read_runs(path: Path = RUNS_PATH) -> list[dict]:
    """Read persisted JSONL evaluation records."""
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

if RUN_BATCH:
    existing = {
        r["question_id"]
        for r in read_runs()
        if matches_experiment(r) and r.get("status") != "erro"
    }
    for question_id in questions["question_id"]:
        if question_id in existing:
            print(f"{question_id}: já registrado; ignorado")
            continue
        record = evaluate_one(question_id)
        if record["status"] != "erro":
            existing.add(question_id)
        print(f"{question_id}: {record['status']} - {record.get('error')} ({record['elapsed_seconds']} s)")
else:
    print("Lote desativado. Altere RUN_BATCH para True quando desejar.")

Lote desativado. Altere RUN_BATCH para True quando desejar.


## 7. Consolidar resultados automáticos
Esta tabela descreve comportamento técnico, **não mede fidelidade semântica**. Perguntas fora do escopo têm abstenção esperada; perguntas vagas exigem julgamento contextual e não são automaticamente classificadas como acerto/erro.

In [105]:
"""Build an execution summary without confusing errors with abstentions."""
runs = read_runs()
latest = {}
for record in runs:
    if matches_experiment(record):
        latest[record["question_id"]] = record
current = [
    r for r in latest.values()
]
summary = pd.DataFrame([{
    "question_id": r["question_id"],
    "scope": r["scope"],
    "status": r["status"],
    "reason": r["insufficiency_reason"],
    "citations": len(r["cited_review_ids"]),
    "retrieved": len(r["retrieval"]["reviews"]),
    "latency_s": r["elapsed_seconds"],
    "out_of_scope_abstention": (
        None if r["status"] == "erro" else r["status"] == "abstencao"
    ) if r["scope"] == "out_of_scope" else None,
} for r in current])
if not summary.empty:
    display(summary.sort_values("question_id"))
    print("Execuções:", len(summary), "| Erros técnicos:", int((summary["status"] == "erro").sum()))
    valid_oos = summary[(summary["scope"] == "out_of_scope") & (summary["status"] != "erro")]
    print("Abstenções fora do escopo:", int((valid_oos["status"] == "abstencao").sum()),
          "/", len(valid_oos), "execuções sem erro")
else:
    print("Nenhuma execução registrada para esta configuração.")

,question_id,scope,status,reason,citations,retrieved,latency_s,out_of_scope_abstention
0,Q01,domain,respondida,nenhuma,16,20,240.402,None
1,Q02,domain,respondida,nenhuma,19,20,48.508,None
2,Q03,domain,respondida,nenhuma,18,20,308.531,None
3,Q04,domain,respondida,nenhuma,2,20,327.382,None
4,Q05,domain,respondida,nenhuma,2,20,25.180,None
5,Q06,domain,respondida,nenhuma,3,20,146.298,None
6,Q07,domain,respondida,nenhuma,7,20,265.725,None
7,Q08,domain,respondida,nenhuma,16,20,67.321,None
8,Q09,domain,respondida,nenhuma,15,20,222.721,None
9,Q10,out_of_scope,abstencao,fora_do_escopo,0,20,399.538,True


Execuções: 19 | Erros técnicos: 0
Abstenções fora do escopo: 7 / 7 execuções sem erro


## 8. Planilha de avaliação humana
Para cada pergunta, julgue manualmente:

- **`faithfulness`**: `2` = todas as afirmações sustentadas; `1` = parcialmente sustentadas; `0` = afirmações não sustentadas.
- **`citation_quality`**: `2` = citações suficientes e corretas; `1` = parcialmente corretas/incompletas; `0` = incorretas; deixar vazio se não houver citações.
- **`answer_quality`**: `2` = síntese clara e responde ao pedido; `1` = parcialmente; `0` = inadequada.
- **`abstention_correct`**: `1` = abstenção apropriada; `0` = inapropriada; vazio quando não aplicável.

A pergunta fora do escopo deve ser recusada; uma pergunta de domínio pode ser corretamente recusada se o contexto não trouxer evidências. Não confunda `status=erro` com abstenção.

A célula cria o CSV **apenas se ainda não existir**, preservando rótulos já atribuídos.

Registros com `status=erro` não são criados como novas linhas de avaliação. Rótulos humanos existentes são preservados e o status técnico é sincronizado para a configuração correspondente.


In [106]:
"""Append new evaluation questions while preserving existing human labels."""
LABEL_COLUMNS = [
    "question_id", "model", "top_k", "prompt_sha256", "status", "faithfulness",
    "citation_quality", "answer_quality", "abstention_correct", "notes",
]
LABEL_KEY = ["question_id", "model", "top_k", "prompt_sha256"]

# Read labels as text so that manual notes and blank ratings are preserved.
if LABELS_PATH.exists():
    labels_existing = pd.read_csv(LABELS_PATH, dtype=str, keep_default_na=False)
    if "prompt_sha256" not in labels_existing.columns:
        labels_existing["prompt_sha256"] = ""  # Legacy rows: prompt unknown.
    missing_columns = set(LABEL_COLUMNS) - set(labels_existing.columns)
    if missing_columns:
        raise ValueError(f"Colunas ausentes no CSV de rótulos: {sorted(missing_columns)}")
    labels_before = labels_existing.copy()
    if LEGACY_PROMPT_CONFIRMED:
        legacy_mask = (
            labels_existing["prompt_sha256"].eq("")
            & labels_existing["model"].eq(MODEL_ID)
            & labels_existing["top_k"].eq(str(TOP_K))
        )
        labels_existing.loc[legacy_mask, "prompt_sha256"] = PROMPT_SHA256
    if labels_existing.duplicated(LABEL_KEY).any():
        raise ValueError("Há perguntas duplicadas para a mesma configuração no CSV de rótulos.")
else:
    labels_existing = pd.DataFrame(columns=LABEL_COLUMNS)
    labels_before = labels_existing.copy()

# Sync only technical status, never overwrite human judgments.
status_by_key = {
    (r["question_id"], r["model"], str(r["top_k"]),
     r.get("prompt_sha256") or (PROMPT_SHA256 if LEGACY_PROMPT_CONFIRMED else "")): r["status"]
    for r in current
}
labels_existing["status"] = [
    status_by_key.get((q, m, str(k), h),
        status_by_key.get((q, m, str(k), PROMPT_SHA256), s)
        if h == "" and LEGACY_PROMPT_CONFIRMED else s)
    for q, m, k, h, s in labels_existing[
        ["question_id", "model", "top_k", "prompt_sha256", "status"]
    ].itertuples(index=False, name=None)
]
existing_keys = {
    tuple(str(value) for value in row)
    for row in labels_existing[LABEL_KEY].itertuples(index=False, name=None)
}
new_rows = []
for record in sorted(current, key=lambda item: item["question_id"]):
    if record["status"] == "erro":
        continue
    prompt_hash = record.get("prompt_sha256") or (PROMPT_SHA256 if LEGACY_PROMPT_CONFIRMED else "")
    key = (record["question_id"], record["model"], str(record["top_k"]), prompt_hash)
    if key in existing_keys:
        continue
    new_rows.append({
        "question_id": record["question_id"],
        "model": record["model"],
        "top_k": str(record["top_k"]),
        "prompt_sha256": prompt_hash,
        "status": record["status"],
        "faithfulness": "",
        "citation_quality": "",
        "answer_quality": "",
        "abstention_correct": "",
        "notes": "",
    })
    existing_keys.add(key)

changed = not labels_existing.equals(labels_before)
if new_rows or changed or not LABELS_PATH.exists():
    updated = pd.concat(
        [labels_existing, pd.DataFrame(new_rows, columns=LABEL_COLUMNS)],
        ignore_index=True,
    )
    updated.to_csv(LABELS_PATH, index=False)
    print(f"Planilha atualizada: {len(new_rows)} pergunta(s) adicionada(s); status sincronizado.")
else:
    print("Planilha preservada: nenhuma pergunta nova a adicionar.")

labels_preview = pd.read_csv(LABELS_PATH, dtype=str, keep_default_na=False)
display(labels_preview.head())
print("Perguntas com resposta ou abstenção para avaliar:",
      int(labels_preview["status"].isin(["respondida", "abstencao"]).sum()))

Planilha preservada: nenhuma pergunta nova a adicionar.


,question_id,model,top_k,status,faithfulness,citation_quality,answer_quality,abstention_correct,notes,prompt_sha256
0,Q01,gemini-3.8-flash,20,respondida,2,2,2,,"Resposta fiel às evidências, 16 citações verif...",a128a1fe07c1
1,Q02,gemini-3.8-flash,20,respondida,2,2,2,,Resposta fiel às evidências; 19 citações perti...,a128a1fe07c1
2,Q03,gemini-3.8-flash,20,respondida,2,2,2,,Resposta fiel; 18 avaliações pertinentes confi...,a128a1fe07c1
3,Q04,gemini-3.8-flash,20,respondida,2,2,2,,Resposta fiel às duas evidências diretas recup...,a128a1fe07c1
4,Q05,gemini-3.8-flash,20,respondida,2,2,2,,"Resposta fiel às duas avaliações citadas, amb...",a128a1fe07c1


Perguntas com resposta ou abstenção para avaliar: 19


## 9. Resumo dos julgamentos humanos
Preencha `evaluation/generation_labels.csv` fora do notebook e reexecute a célula. Só são exibidas médias dos valores efetivamente preenchidos; campos vazios não são tratados como zero. **Não interpretar médias parciais como avaliação final das 19 perguntas.**

In [107]:
"""Summarize only manually completed ratings, without inventing scores."""
labels = pd.read_csv(LABELS_PATH, dtype={"prompt_sha256": str})
for col in ["faithfulness", "citation_quality", "answer_quality", "abstention_correct"]:
    labels[col] = pd.to_numeric(labels[col], errors="coerce")
    valid = {0, 1} if col == "abstention_correct" else {0, 1, 2}
    assert labels[col].dropna().isin(valid).all(), f"Rótulo inválido em {col}"
ratings = pd.DataFrame([
    {"metric": col, "labeled": int(labels[col].notna().sum()),
     "mean": labels[col].mean()}
    for col in ["faithfulness", "citation_quality", "answer_quality", "abstention_correct"]
])
display(ratings)
completed = sum(r["status"] != "erro" for r in current)
judgable = int(labels["status"].isin(["respondida", "abstencao"]).sum())
print(f"Perguntas com execução válida: {completed}/19 | Linhas avaliáveis: {judgable}/19")

,metric,labeled,mean
0,faithfulness,9,2.0
1,citation_quality,9,2.0
2,answer_quality,9,2.0
3,abstention_correct,10,1.0


Perguntas com execução válida: 19/19 | Linhas avaliáveis: 19/19


## 10. Auditoria das evidências (sem chamadas ao Gemini)

Selecione uma pergunta de `Q01` a `Q19` em `AUDIT_QUESTION_ID` e execute a célula abaixo. Ela lê **somente** os arquivos `evaluation/generation_runs.jsonl` e `evaluation/generation_labels.csv`: apresenta a última execução válida da configuração confirmada, a resposta original, as avaliações citadas em texto integral e o julgamento humano.

O histórico de tentativas com erro permanece no JSONL, mas não é confundido com a execução válida. `id_found` indica apenas a existência do identificador no snapshot recuperado; **a pertinência semântica da citação exige inspeção humana**. Registros legados sem hash só são aceitos quando `LEGACY_PROMPT_CONFIRMED=True`.

Esta seção não executa `generate_answer`, não consulta o ChromaDB e não consome cota da API. A reprodução de uma nova resposta é um procedimento distinto da auditoria da execução original.

In [108]:
"""Audit saved generation evidence and human labels without API calls."""
AUDIT_QUESTION_ID = "Q01"  # Change to Q02 ... Q19.


def audit_saved_question(question_id: str) -> None:
    """Display the latest valid saved run, cited reviews and manual judgment."""
    allowed = {f"Q{i:02d}" for i in range(1, 20)}
    if question_id not in allowed:
        raise ValueError(f"Pergunta inválida: {question_id}")

    matching = [
        run for run in read_runs()
        if run.get("question_id") == question_id and matches_experiment(run)
    ]
    valid_runs = [run for run in matching if run.get("status") in ("respondida", "abstencao")]
    if not valid_runs:
        print(f"Sem execução válida para {question_id} na configuração atual.")
        return

    # Keep file append order, preserving the original historical snapshot.
    run = valid_runs[-1]
    print(f"PERGUNTA: {run['question']}")
    print(f"MODELO: {run['model']} | TOP_K: {run['top_k']}")
    print(f"PROMPT: {run.get('prompt_sha256') or 'legado, equivalência confirmada por configuração'}")
    print(f"DATA UTC: {run.get('timestamp_utc', 'não registrada')}")
    print(f"STATUS: {run['status']} | MOTIVO: {run['insufficiency_reason']}")
    print(f"TENTATIVAS HISTÓRICAS COMPATÍVEIS: {len(matching)}")
    print(f"\nRESPOSTA ORIGINAL:\n{run['answer']}")

    retrieved = run.get("retrieval", {}).get("reviews", [])
    by_id = {review["review_id"]: review for review in retrieved}
    citations = run.get("cited_review_ids", [])
    print(f"\nEVIDÊNCIAS CITADAS ({len(citations)}):")
    if not citations:
        print("Nenhuma (abstenção)." if run['status'] == 'abstencao' else "Nenhuma.")
    for review_id in citations:
        review = by_id.get(review_id)
        if review is None:
            print(f"[ID AUSENTE NO SNAPSHOT] {review_id}")
        else:
            print(f"\n[{review_id}] {review['document']}")
    print(f"\nINTEGRIDADE DAS CITAÇÕES: {sum(x in by_id for x in citations)}/{len(citations)} IDs encontrados")

    if not LABELS_PATH.exists():
        print("\nCSV de julgamentos humanos não encontrado.")
        return
    labels_audit = pd.read_csv(LABELS_PATH, dtype=str, keep_default_na=False)
    if "prompt_sha256" not in labels_audit.columns:
        labels_audit["prompt_sha256"] = ""
    label_match = labels_audit.loc[
        labels_audit["question_id"].eq(question_id)
        & labels_audit["model"].eq(MODEL_ID)
        & labels_audit["top_k"].eq(str(TOP_K))
        & (
            labels_audit["prompt_sha256"].eq(PROMPT_SHA256)
            | (LEGACY_PROMPT_CONFIRMED & labels_audit["prompt_sha256"].eq(""))
        )
    ]
    if len(label_match) != 1:
        print(f"\nJULGAMENTO HUMANO: {len(label_match)} linhas compatíveis (esperada 1).")
        return
    label = label_match.iloc[0]
    print("\nJULGAMENTO HUMANO:")
    for field in ("faithfulness", "citation_quality", "answer_quality", "abstention_correct", "notes"):
        print(f"{field}: {label.get(field, '') or 'não aplicável/não preenchido'}")


audit_saved_question(AUDIT_QUESTION_ID)

PERGUNTA: Quais avaliações relatam que o produto não foi entregue?
MODELO: gemini-3.8-flash | TOP_K: 20
PROMPT: legado, equivalência confirmada por configuração
DATA UTC: 2026-11-11T01:20:56.994837+00:00
STATUS: respondida | MOTIVO: nenhuma
TENTATIVAS HISTÓRICAS COMPATÍVEIS: 1

RESPOSTA ORIGINAL:
Diversos clientes relatam expressamente que seus pedidos não foram entregues ou não haviam chegado até o momento da avaliação. Além de apontarem a ausência de recebimento da mercadoria, alguns consumidores cobram explicações sobre o ocorrido e buscam uma solução para o problema. Há também relatos de frustração por serem solicitados a avaliar uma compra antes mesmo de tê-la em mãos.

EVIDÊNCIAS CITADAS (16):

[c84ca514000aa704df0258d693a07414] Por favor, peço que chequem o porque do produto não ter sido entregue.

[09db4c326e7158211d66b9f994542274] PRODUTO NÃO ENTREGUE Gostaria de saber pq o produto não foi entregue. Gostaria de uma solução

[4d05593294cb87f9412ec522d3c6333e] ainda não recebi o

## 11. Conclusões da avaliação de geração

**Configuração avaliada:** Gemini (`gemini-3.8-flash`, camada gratuita), `top_k=20`, `temperature=0`, prompt `a128a1fe07c1`, 19 perguntas com uma execução cada. Os resultados descrevem esta configuração e esta amostra.

### 11.1. Resultados

**Execução e abstenção**

| Grupo | Perguntas | Resultado |
|---|---:|---|
| Domínio | 9 | 9 respondidas |
| Fora do escopo | 7 | 7 abstenções (6 `fora_do_escopo`, 1 `evidencia_insuficiente`) |
| Vagas | 3 | 3 abstenções (`pergunta_vaga`) |
| Erros técnicos nas execuções finais | 19 | 0 |

Todas as 19 perguntas têm execução válida. Na primeira rodada houve falhas de API (Q11: 503; Q18 e Q19: 429, cota excedida) e Q18 falhou mais duas vezes com 503 antes de concluir. As falhas ficaram registradas como `erro` e nunca foram contadas como abstenção.

**Citações contra os rótulos de recuperação (Q01–Q09)**
Métrica automática: compara os IDs citados com os rótulos humanos (0/1/2) do Notebook 02. As 20 avaliações recuperadas de cada pergunta estavam todas rotuladas.

| Pergunta | Citadas | Precisão estrita | Cobertura (rótulo 2) |
|---|---:|---:|---:|
| Q01 | 16 | 16/16 | 16/16 |
| Q02 | 19 | 18/19 | 18/19 |
| Q03 | 18 | 18/18 | 18/18 |
| Q04 | 2 | 2/2 | **2/5** |
| Q05 | 2 | 2/2 | 2/2 |
| Q06 | 3 | 3/3 | 3/3 |
| Q07 | 7 | 7/7 | **7/12** |
| Q08 | 16 | 16/16 | 16/18 |
| Q09 | 15 | 15/15 | 15/15 |
| **Total** | **98** | **97/98 (99,0%)** | **97/108 (89,8%)** |

**Avaliação manual (nove respostas fundamentadas e dez abstenções)**

| Métrica | Resultado |
|---|---:|
| Fidelidade média às evidências | 2,00/2 |
| Qualidade média das citações | 2,00/2 |
| Qualidade média das respostas | 2,00/2 |
| Abstenções consideradas adequadas | 10/10 |

A nota de `citation_quality` avalia se as avaliações citadas são pertinentes e sustentam a resposta. A **cobertura** das citações é medida à parte, na tabela anterior, e não entra nessa nota.

**Latência:** de 25 s a 1854 s por pergunta, na camada gratuita da API.

### 11.2. Principais observações

**Citações precisas.** Das 98 avaliações citadas, 97 têm rótulo 2 e nenhuma tem rótulo 0. A única exceção é uma citação de rótulo 1 em Q02. O sistema não precisou citar todos os documentos recuperados.

**Filtro por sentido do LLM.** Na recuperação, Q04–Q06 tinham precisão estrita de 10% a 27% em k=30, porque o embedding captura o tema, mas não o sentido. Com o LLM selecionando, as citações dessas perguntas são todas de rótulo 2. Em Q05, o modelo citou 2 das 20 recuperadas e descartou as 15 que dizem "conforme anunciado". Isso confirma a decisão de recuperar com k=20 e deixar o LLM filtrar pelo sentido.

**Abstenção por julgamento do LLM, não por similaridade.** Q16 tinha similaridade de 0,738 no Notebook 02, maior que a de Q06 e Q07 (perguntas de domínio), e mesmo assim foi recusada. Isso sustenta a decisão de não usar um limiar de similaridade. O modelo também pediu esclarecimento nas perguntas vagas.

**Fidelidade das respostas.** Na leitura das nove respostas, as afirmações são sustentadas pelas avaliações citadas. Uma expressão em Q07 é ambígua: "recusa no recebimento de produtos danificados", quando a avaliação diz "Produto danificado e não aceita devolução" (recusa de devolução pela empresa). Pode ser lida também como o consumidor recusando o recebimento.

**Cobertura irregular.** O modelo privilegia precisão sobre cobertura. Em Q04 omitiu 3 das 5 avaliações de rótulo 2, entre elas "Produto com avarias e péssima qualidade", que descreve dano de forma explícita. Em Q07 omitiu 5 das 12, incluindo pedidos de reembolso sem resposta. A média de 89,8% esconde essa dispersão: cinco perguntas ficam em 100% ou perto disso.

### 11.3. Limitações identificadas

1. **Cobertura limitada em Q04 e Q07** (2/5 e 7/12). O sistema pode subestimar a frequência de um problema ao omitir avaliações pertinentes.
2. **Cobertura relativa ao top-20.** Avaliações relevantes fora do top-20 não entram nessa conta; a fidelidade das respostas não garante que todas as avaliações pertinentes da base foram recuperadas (ver Notebook 02).
3. **Avaliação manual por uma única pessoa, com uma execução por pergunta.** As notas ficaram no teto (2/2/2), o que limita a capacidade de discriminar qualidade, e não há segundo avaliador para estimar concordância. `temperature=0` não garante determinismo total no Gemini.
4. **Amostra:** nove perguntas de domínio, todas no mesmo molde ("Quais avaliações…", "Existem relatos…"). Os resultados não devem ser extrapolados para consultas livres.
5. **Fronteira de critério em Q02:** a citação de rótulo 1 e a omissão de "Recomendo o produto e o fornecedor, pois o atraso na entrega foi causado pelos serviços dos Correios" vêm da diferença entre mencionar atraso e reclamar de atraso.
6. **Classificação do motivo de abstenção:** em Q14 a abstenção foi correta, mas o motivo registrado foi `evidencia_insuficiente`, quando `fora_do_escopo` seria mais adequado.
7. **Perguntas amplas:** em Q18 ("Problemas com a compra") o modelo se absteve, o que está de acordo com as instruções atuais. Uma configuração exploratória poderia oferecer um panorama dos problemas encontrados.
8. **Consultas curtas:** Q17 e Q19 recuperaram apenas 4 e 2 avaliações (`retrieval_incomplete=True`), formadas por textos de uma palavra ("Pontual!", "Bom !", "Boa Recomendo"), que a deduplicação colapsou.
9. **Infraestrutura:** falhas de API (503 e 429) e latência de até 1854 s na camada gratuita. Sem comparação entre modelos.

### 11.4. Rastreabilidade e auditoria
- Os resultados originais estão em `evaluation/generation_runs.jsonl` (perguntas, respostas, citações, avaliações recuperadas e histórico de tentativas, inclusive as com erro).
- Os julgamentos manuais estão em `evaluation/generation_labels.csv`, com notas individuais.
- A seção 10 permite auditar cada pergunta sem chamar o Gemini nem reconstruir o índice.

### 11.5. Conclusão geral

No conjunto de 19 perguntas e com avaliação de uma única anotadora, os resultados são compatíveis com geração fundamentada e abstenção adequada. A precisão das citações foi alta (99,0%); a cobertura foi menor (89,8%) e irregular entre perguntas. O desempenho em consultas fora do conjunto testado não foi avaliado.

**Decisões:** manter `top_k=20` e o prompt `a128a1fe07c1` como linha de base; manter a suficiência decidida por julgamento do LLM e por regras determinísticas, sem limiar de similaridade; registrar a baixa cobertura em Q04 e Q07 como limitação conhecida.

**Próximo experimento, se necessário:** comparar `top_k=20` com `top_k=30`, mantendo modelo, perguntas e critérios fixos, para verificar se a cobertura de Q04 e Q07 melhora. Os rótulos de recuperação existentes cobrem até o ranking 30.

Com a avaliação concluída e as limitações registradas, o projeto pode avançar para a interface de consulta.